# Protein Structure Prediction with the AlphaFold2 NIM
This example notebook demonstrates how deploy the NVIDIA AlphaFold2 NIM for predicting the 3D structure of a protein from its amino acid sequence.

NVIDIA BioNeMo NIMS can be integrated into existing virtual screening workflows to leverage cutting edge Gen-AI capabilities for drug discovery, from ligand generation to protein folding to docking. For more details, please visit the [NVIDIA BioNeMo NIMS homepage](https://build.nvidia.com/explore/healthcare).

These capabilities are also integrated into reference workflows in NVIDIA NIM Agent Blueprints. For more details, please visit [NVIDIA NIM Blueprints](https://build.nvidia.com/nim/blueprints).


## Setup
Please make sure the required packages and dependencies are installed. You may install the required packages using the following `pip install` command.

In [15]:
!ls -l /opt/nim/.cache

lrwxrwxrwx 1 root root 35 Apr  5 09:57 /opt/nim/.cache -> /home/ec2-user/SageMaker/nim/.cache


**--> check the cache directory if cache files are there and rights are correct**

In [26]:
ls -l /home/ec2-user/SageMaker/.cache/nim

total 8
drwxr-xr-x 9 ec2-user ec2-user 4096 Mar 17 12:54 alphafold2-data_v1.0.0/
drwxr-xr-x 3 ec2-user ec2-user 4096 Apr  5 19:11 ngc/


In [28]:
import boto3
import json

# Replace with your secret name and region
secret_name = 'NVIDIA_NGC_CREDS_VINCENT'
region_name = 'us-east-1'

# Create a Secrets Manager client
client = boto3.client('secretsmanager', region_name=region_name)

# Retrieve the secret value
response = client.get_secret_value(SecretId=secret_name)
secret = json.loads(response['SecretString'])

# Store the API key in a Python variable
ngc_cli_api_key = secret['NGC_CLI_API_KEY']

# (Optional) Print a success message (avoid printing the actual key)
print("API key retrieved successfully.")

API key retrieved successfully.


In [29]:
%%bash -s "$ngc_cli_api_key"
# Set the API key as an environment variable
export NGC_CLI_API_KEY=$1

#redicrec local nim cache ###
#sudo ln -s /home/ec2-user/SageMaker/docker /var/lib/docker #--> Should be done already!
#sudo ln -s /home/ec2-user/SageMaker/nim/.cache /opt/nim/.cache #--> Should be done already!

# docker login
docker login nvcr.io --username '$oauthtoken' --password $NGC_CLI_API_KEY

# Authenticate to your ECR registry
aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 841162696343.dkr.ecr.us-east-1.amazonaws.com

#export path to Local nim path var
export LOCAL_NIM_CACHE=/home/ec2-user/SageMaker/.cache/nim

#run the container
docker run -d --rm --name af2 --runtime=nvidia \
        -e CUDA_VISIBLE_DEVICES=0 \
        -e NGC_API_KEY=$NGC_CLI_API_KEY \
        -v $LOCAL_NIM_CACHE:/opt/nim/.cache \
        -p 8081:8000 \
        841162696343.dkr.ecr.us-east-1.amazonaws.com/alphafold2:latest

WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
ab012f1dad862da58bf6e559303485de0e9bbcc76eb20cbb680207845f8b0a42


--> original code if needed just as RAW

In [27]:
!docker ps -a
# In case you want to stop the container -> !docker stop alphafold2

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


# Stop AF2 Container

In [21]:
!docker stop af2

af2


# Code Store